In [1]:
# import os
# for root, dirs, files in os.walk("/kaggle/input/datasets/"):
#     for file in files:
#         print(os.path.join(root, file))


In [2]:
import pickle
import numpy as np

BASE = "/kaggle/input/datasets/ayushaabbas/patchcore-outputs/kaggle/working"

with open(f"{BASE}/patchcore_full_results.pkl", "rb") as f:
    results = pickle.load(f)

memory_banks = {}
for cat in results.keys():
    memory_banks[cat] = np.load(f"{BASE}/membank_{cat}.npy")

print("✓ Categories:", list(results.keys()))
print("✓ Memory banks:", {k: v.shape for k, v in memory_banks.items()})
print("✓ Sample path:", results['bottle']['image_paths'][:2])


✓ Categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
✓ Memory banks: {'bottle': (1638, 1536), 'cable': (1756, 1536), 'capsule': (1716, 1536), 'carpet': (2195, 1536), 'grid': (2069, 1536), 'hazelnut': (3065, 1536), 'leather': (1920, 1536), 'metal_nut': (1724, 1536), 'pill': (2093, 1536), 'screw': (2508, 1536), 'tile': (1803, 1536), 'toothbrush': (470, 1536), 'transistor': (1669, 1536), 'wood': (1936, 1536), 'zipper': (1881, 1536)}
✓ Sample path: ['/kaggle/input/datasets/ipythonx/mvtec-ad/bottle/test/broken_large/000.png'
 '/kaggle/input/datasets/ipythonx/mvtec-ad/bottle/test/broken_large/001.png']


In [3]:
from PIL import Image
import torch
import torch.nn as nn
import timm
import numpy as np
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone = timm.create_model('wide_resnet50_2', pretrained=True, features_only=True)
backbone = backbone.to(device)
backbone.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("✓ Backbone ready")


model.safetensors:   0%|          | 0.00/276M [00:00<?, ?B/s]

✓ Backbone ready


In [4]:
def extract_features(image_path):
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = backbone(img_tensor)
    f2 = features[2]
    f3 = features[3]
    f3_up = nn.functional.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
    combined = torch.cat([f2, f3_up], dim=1)
    b, c, h, w = combined.shape
    patches = combined.permute(0, 2, 3, 1).reshape(-1, c)
    return patches.cpu().numpy()


In [5]:
def score_image(image_path, nn_index, k=9):
    patches = extract_features(image_path)
    distances, _ = nn_index.kneighbors(patches)
    anomaly_map = distances.mean(axis=1).reshape(28, 28)
    image_score = anomaly_map.max()
    return image_score, anomaly_map


In [6]:
import pickle
import numpy as np

BASE = "/kaggle/input/datasets/ayushaabbas/patchcore-outputs/kaggle/working"

with open(f"{BASE}/patchcore_full_results.pkl", "rb") as f:
    results = pickle.load(f)

memory_banks = {}
for cat in results.keys():
    memory_banks[cat] = np.load(f"{BASE}/membank_{cat}.npy")

print("✓ Results loaded")
print("✓ Memory banks loaded")


✓ Results loaded
✓ Memory banks loaded


In [7]:
"""
HITL Three-Strategy Experiment — Single Category: CABLE
=====================================================================
Strategies: passive (random), active (uncertainty), geometric (centroid-aware)
Saves geo_centroid_dist per round for geometric analysis.
"""

import numpy as np
import pickle
import os
import time
from copy import deepcopy
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support,
                              confusion_matrix, roc_curve,
                              average_precision_score)

# ── CONFIG ── only change this line per notebook ──────────────
CATEGORY   = 'cable'
N_ROUNDS   = 30
SEED       = 42
SAVE_DIR   = "/kaggle/working"
STRATEGIES = ['geometric']


# ── HELPERS ───────────────────────────────────────────────────

def build_nn(memory_bank, k=9):
    nn_idx = NearestNeighbors(n_neighbors=k, metric='euclidean',
                              algorithm='ball_tree', n_jobs=-1)
    nn_idx.fit(memory_bank)
    return nn_idx


def score_all(image_paths, nn_index):
    scores = []
    for path in image_paths:
        s, _ = score_image(str(path), nn_index)
        scores.append(s)
    return np.array(scores)


def compute_metrics_at_threshold(labels, scores, threshold):
    preds = (scores >= threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'threshold': float(threshold),
        'precision': float(prec),
        'recall':    float(rec),
        'f1':        float(f1),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'accuracy':  float((tp + tn) / (tp + tn + fp + fn))
    }


def apply_fp_correction(img_path, memory_bank, nn_index, max_bank_size=2500):
    new_patches = extract_features(str(img_path))
    memory_bank = np.concatenate([memory_bank, new_patches], axis=0)
    if len(memory_bank) > max_bank_size:
        indices = np.random.choice(len(memory_bank), max_bank_size, replace=False)
        memory_bank = memory_bank[indices]
    nn_index = build_nn(memory_bank)
    return memory_bank, nn_index, len(new_patches)


def apply_fn_correction(img_path, memory_bank, nn_index, k_remove=5):
    patches   = extract_features(str(img_path))
    dists, _  = nn_index.kneighbors(patches, n_neighbors=1)
    anchor    = patches[np.argmax(dists.squeeze())].reshape(1, -1)
    k_actual  = min(k_remove, len(memory_bank) - 10)
    _, rm_idx = nn_index.kneighbors(anchor, n_neighbors=k_actual)
    keep_mask = np.ones(len(memory_bank), dtype=bool)
    keep_mask[rm_idx.flatten()] = False
    n_removed   = int((~keep_mask).sum())
    memory_bank = memory_bank[keep_mask]
    nn_index    = build_nn(memory_bank)
    return memory_bank, nn_index, n_removed


def compute_aucc(auroc_curve, baseline_auroc):
    gains = [max(0.0, a - baseline_auroc) for a in auroc_curve[1:]]
    if not gains or max(gains) == 0:
        return 0.0
    max_possible = len(gains) * (1.0 - baseline_auroc)
    return float(np.trapezoid(gains) / max_possible) if max_possible > 0 else 0.0


# ── GEOMETRIC QUERY STRATEGY ──────────────────────────────────

def select_geometric(fp_pool, fn_pool, memory_bank, pool_cap=20):
    """
    Geometry-aware query strategy based on memory bank centroid distance.

    FP: select the normal image whose patches are most distant from
    the memory bank centroid — fills the largest gap in normal coverage.

    FN: select the anomalous image whose most anomalous patch is
    closest to the bank centroid — removes the most centrally
    confusing normal features.

    Returns: (selected_candidate, geo_score) where geo_score is the
    centroid distance that drove the selection — saved for analysis.
    """
    centroid = memory_bank.mean(axis=0)
    bank_std = memory_bank.std() + 1e-8

    fp_candidates = sorted(fp_pool, key=lambda x: x[0], reverse=True)[:pool_cap]
    fn_candidates = sorted(fn_pool, key=lambda x: x[0], reverse=True)[:pool_cap]

    # Score FP candidates — highest mean patch distance to centroid wins
    best_fp       = None
    best_fp_score = -np.inf
    for candidate in fp_candidates:
        score_val, img_path, ctype, subtype = candidate
        patches = extract_features(str(img_path))
        geo_score = np.linalg.norm(patches - centroid, axis=1).mean()
        if geo_score > best_fp_score:
            best_fp_score = geo_score
            best_fp = candidate

    # Score FN candidates — lowest anchor distance to centroid wins
    best_fn       = None
    best_fn_score = np.inf
    if fn_candidates:
        nn_tmp = NearestNeighbors(n_neighbors=1, metric='euclidean', algorithm='ball_tree')
        nn_tmp.fit(memory_bank)
        for candidate in fn_candidates:
            score_val, img_path, ctype, subtype = candidate
            patches = extract_features(str(img_path))
            dists, _ = nn_tmp.kneighbors(patches, n_neighbors=1)
            anchor = patches[np.argmax(dists.squeeze())]
            dist_to_centroid = np.linalg.norm(anchor - centroid)
            if dist_to_centroid < best_fn_score:
                best_fn_score = dist_to_centroid
                best_fn = candidate

    # Select between FP and FN using normalised priority scores
    if best_fp is not None and best_fn is not None:
        fp_priority = best_fp_score / bank_std
        fn_priority = bank_std / (best_fn_score + 1e-8)
        if fp_priority >= fn_priority:
            return best_fp, float(best_fp_score)
        else:
            return best_fn, float(best_fn_score)
    elif best_fp is not None:
        return best_fp, float(best_fp_score)
    elif best_fn is not None:
        return best_fn, float(best_fn_score)
    return None, None


# ── MAIN FEEDBACK LOOP ────────────────────────────────────────

def run_feedback_loop(category, strategy, n_rounds=N_ROUNDS, seed=SEED):
    rng        = np.random.default_rng(seed)
    start_time = time.time()

    r           = results[category]
    image_paths = r['image_paths']
    labels      = r['labels']
    subtypes    = r['subtypes']
    memory_bank = deepcopy(memory_banks[category])
    nn_index    = build_nn(memory_bank)

    # ── Baseline ──────────────────────────────────────────────
    scores           = score_all(image_paths, nn_index)
    baseline_auroc   = roc_auc_score(labels, scores)
    threshold        = np.percentile(scores[labels == 0], 90)
    baseline_ap      = average_precision_score(labels, scores)
    baseline_metrics = compute_metrics_at_threshold(labels, scores, threshold)
    fpr, tpr, _      = roc_curve(labels, scores)

    print(f"    Baseline AUROC: {baseline_auroc:.4f}  AP: {baseline_ap:.4f}")

    auroc_curve         = [baseline_auroc]
    ap_curve            = [baseline_ap]
    f1_curve            = [baseline_metrics['f1']]
    precision_curve     = [baseline_metrics['precision']]
    recall_curve        = [baseline_metrics['recall']]
    threshold_curve     = [float(threshold)]
    n_corrections       = [0]
    n_fp_corrections    = [0]
    n_fn_corrections    = [0]
    bank_size_curve     = [len(memory_bank)]
    scores_per_round    = [scores.copy()]
    correction_log      = []
    round_times         = []

    corrected = set()
    total_fp  = 0
    total_fn  = 0

    for round_idx in range(n_rounds):
        t0 = time.time()

        scores    = score_all(image_paths, nn_index)
        threshold = np.percentile(scores[labels == 0], 90)

        fp_pool = [(scores[i], image_paths[i], 'FP', subtypes[i])
                   for i in range(len(image_paths))
                   if labels[i] == 0 and scores[i] >= threshold
                   and str(image_paths[i]) not in corrected]

        fn_pool = [(scores[i], image_paths[i], 'FN', subtypes[i])
                   for i in range(len(image_paths))
                   if labels[i] == 1 and scores[i] < threshold
                   and str(image_paths[i]) not in corrected]

        pool = fp_pool + fn_pool
        if not pool:
            print(f"    Round {round_idx+1:3d}: no errors remain — stopping early")
            final_val = auroc_curve[-1]
            while len(auroc_curve) < n_rounds + 1:
                auroc_curve.append(final_val)
                ap_curve.append(ap_curve[-1])
                f1_curve.append(f1_curve[-1])
                precision_curve.append(precision_curve[-1])
                recall_curve.append(recall_curve[-1])
                threshold_curve.append(threshold_curve[-1])
                n_corrections.append(n_corrections[-1])
                n_fp_corrections.append(n_fp_corrections[-1])
                n_fn_corrections.append(n_fn_corrections[-1])
                bank_size_curve.append(bank_size_curve[-1])
                scores_per_round.append(scores_per_round[-1])
                round_times.append(0.0)
            break

        # ── Select correction target ───────────────────────────
        geo_centroid_dist = None  # only set for geometric strategy

        if strategy == 'passive':
            idx = rng.integers(len(pool))
            score_val, img_path, ctype, subtype = pool[idx]

        elif strategy == 'active':
            pool_sorted = sorted(pool, key=lambda x: abs(x[0] - threshold))
            score_val, img_path, ctype, subtype = pool_sorted[0]

        elif strategy == 'geometric':
            selected, geo_centroid_dist = select_geometric(fp_pool, fn_pool, memory_bank)
            if selected is None:
                # Fallback to passive if geometric returns nothing
                idx = rng.integers(len(pool))
                selected = pool[idx]
                geo_centroid_dist = None
            score_val, img_path, ctype, subtype = selected

        # ── Apply correction ───────────────────────────────────
        if ctype == 'FP':
            memory_bank, nn_index, n_patches = apply_fp_correction(
                img_path, memory_bank, nn_index)
            total_fp += 1
        else:
            memory_bank, nn_index, n_patches = apply_fn_correction(
                img_path, memory_bank, nn_index)
            total_fn += 1

        corrected.add(str(img_path))

        # ── Evaluate ──────────────────────────────────────────
        scores     = score_all(image_paths, nn_index)
        curr_auroc = roc_auc_score(labels, scores)
        curr_ap    = average_precision_score(labels, scores)
        new_thresh = np.percentile(scores[labels == 0], 90)
        curr_met   = compute_metrics_at_threshold(labels, scores, new_thresh)
        round_time = time.time() - t0

        # ── Log — includes geo_centroid_dist for geometric ────
        correction_log.append({
            'round':                 round_idx + 1,
            'strategy':              strategy,
            'type':                  ctype,
            'image_path':            str(img_path),
            'subtype':               str(subtype),
            'score_before':          float(score_val),
            'threshold':             float(threshold),
            'distance_to_threshold': float(abs(score_val - threshold)),
            'auroc_after':           float(curr_auroc),
            'auroc_delta':           float(curr_auroc - auroc_curve[-1]),
            'n_patches_changed':     n_patches,
            'bank_size_after':       len(memory_bank),
            'round_time_s':          round_time,
            'geo_centroid_dist':     geo_centroid_dist,  # None for passive/active
        })

        auroc_curve.append(curr_auroc)
        ap_curve.append(curr_ap)
        f1_curve.append(curr_met['f1'])
        precision_curve.append(curr_met['precision'])
        recall_curve.append(curr_met['recall'])
        threshold_curve.append(float(new_thresh))
        n_corrections.append(len(corrected))
        n_fp_corrections.append(total_fp)
        n_fn_corrections.append(total_fn)
        bank_size_curve.append(len(memory_bank))
        scores_per_round.append(scores.copy())
        round_times.append(round_time)

        print(f"    Round {round_idx+1:3d} | {strategy:10s} | "
              f"{ctype} ({str(subtype):<20}) | "
              f"AUROC: {curr_auroc:.4f} | "
              f"corrections: {len(corrected):3d} | "
              f"bank: {len(memory_bank)}")

    total_time = time.time() - start_time
    aucc       = compute_aucc(auroc_curve, baseline_auroc)
    fpr_final, tpr_final, _ = roc_curve(labels, scores_per_round[-1])

    print(f"\n    ── Summary ──")
    print(f"    AUCC:        {aucc:.4f}")
    print(f"    Final AUROC: {auroc_curve[-1]:.4f}  (was {baseline_auroc:.4f})")
    print(f"    Improvement: {auroc_curve[-1] - baseline_auroc:+.4f}")
    print(f"    FP corrected: {total_fp}  FN corrected: {total_fn}")
    print(f"    Total time:   {total_time/60:.1f} min")

    return {
        # ── Core curves ───────────────────────────────────────
        'auroc_curve':        auroc_curve,
        'ap_curve':           ap_curve,
        'f1_curve':           f1_curve,
        'precision_curve':    precision_curve,
        'recall_curve':       recall_curve,
        'threshold_curve':    threshold_curve,
        'n_corrections':      n_corrections,
        'n_fp_corrections':   n_fp_corrections,
        'n_fn_corrections':   n_fn_corrections,
        'bank_size_curve':    bank_size_curve,

        # ── Summary stats ─────────────────────────────────────
        'baseline_auroc':     baseline_auroc,
        'baseline_ap':        baseline_ap,
        'baseline_metrics':   baseline_metrics,
        'final_auroc':        auroc_curve[-1],
        'final_ap':           ap_curve[-1],
        'improvement':        auroc_curve[-1] - baseline_auroc,
        'aucc':               aucc,

        # ── ROC curves ────────────────────────────────────────
        'roc_fpr_baseline':   fpr,
        'roc_tpr_baseline':   tpr,
        'roc_fpr_final':      fpr_final,
        'roc_tpr_final':      tpr_final,

        # ── Per-round raw scores ──────────────────────────────
        'scores_per_round':   scores_per_round,

        # ── Correction log (includes geo_centroid_dist) ───────
        'correction_log':     correction_log,

        # ── Metadata ──────────────────────────────────────────
        'category':           category,
        'strategy':           strategy,
        'n_rounds_run':       len(auroc_curve) - 1,
        'total_fp_corrected': total_fp,
        'total_fn_corrected': total_fn,
        'total_time_s':       total_time,
        'round_times_s':      round_times,
        'image_paths':        image_paths,
        'labels':             labels,
        'subtypes':           subtypes,
    }


# ── RUN ───────────────────────────────────────────────────────

save_path = os.path.join(SAVE_DIR, f"hitl_geo_{CATEGORY}.pkl")

# Auto-resume: check which strategies already done
if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        cat_results = pickle.load(f)
    print(f"✓ Loaded existing results for {CATEGORY}")
    strategies_done = list(cat_results.keys())
    print(f"  Strategies already done: {strategies_done}")
else:
    cat_results = {}
    strategies_done = []

print(f"\n{'='*60}")
print(f"  {CATEGORY.upper()}")
print(f"{'='*60}")

for strategy in STRATEGIES:
    if strategy in strategies_done:
        print(f"\n  Skipping {strategy} (already complete)")
        continue
    print(f"\n  Strategy: {strategy.upper()}")
    cat_results[strategy] = run_feedback_loop(CATEGORY, strategy)

    # Save after every strategy — so timeout only loses current strategy
    with open(save_path, "wb") as f:
        pickle.dump(cat_results, f)
    print(f"  ✓ Saved hitl_geo_{CATEGORY}.pkl")

print(f"\n✓ All strategies complete for {CATEGORY}")


# ── SUMMARY TABLE ─────────────────────────────────────────────

print(f"\n{'='*85}")
print(f"{'Category':<15} {'Strategy':<12} {'Baseline':>9} {'Final':>7} "
      f"{'Improve':>9} {'AUCC':>7} {'FP':>5} {'FN':>5}")
print(f"{'='*85}")

for strategy in STRATEGIES:
    if strategy not in cat_results:
        continue
    r = cat_results[strategy]
    print(f"{CATEGORY:<15} {strategy:<12} {r['baseline_auroc']:>9.4f} "
          f"{r['final_auroc']:>7.4f} {r['improvement']:>+9.4f} "
          f"{r['aucc']:>7.4f} "
          f"{r['total_fp_corrected']:>5} {r['total_fn_corrected']:>5}")




  CABLE

  Strategy: GEOMETRIC
    Baseline AUROC: 0.9185  AP: 0.9542
    Round   1 | geometric  | FP (good                ) | AUROC: 0.9114 | corrections:   1 | bank: 2500
    Round   2 | geometric  | FP (good                ) | AUROC: 0.8958 | corrections:   2 | bank: 2500
    Round   3 | geometric  | FP (good                ) | AUROC: 0.8842 | corrections:   3 | bank: 2500
    Round   4 | geometric  | FP (good                ) | AUROC: 0.8849 | corrections:   4 | bank: 2500
    Round   5 | geometric  | FP (good                ) | AUROC: 0.8855 | corrections:   5 | bank: 2500
    Round   6 | geometric  | FP (good                ) | AUROC: 0.8722 | corrections:   6 | bank: 2500
    Round   7 | geometric  | FP (good                ) | AUROC: 0.8668 | corrections:   7 | bank: 2500
    Round   8 | geometric  | FP (good                ) | AUROC: 0.8656 | corrections:   8 | bank: 2500
    Round   9 | geometric  | FP (good                ) | AUROC: 0.8724 | corrections:   9 | bank: 2500
  